# 🎯 G-LRAG Retrieval — Recall-First, Document-Anchored Edition (F2)

This notebook runs the **redesigned** retrieval pipeline that targets the actual
bottleneck of the F2 grader: **article-level recall**.

## Why this differs from the decomposition notebook

The grader reduces every `law_id|ten_van_ban|Điều X` entry to its bare **`Điều X`
number** and macro-averages Precision / Recall / F2 (β=2). Measured on the dev
set, the old pipeline scored **F2=0.067** with **18/20 questions returning zero
correct articles** — not a precision problem, a *recall* problem: BM25 finds the
right document (doc-recall@50 ≈ 0.75) but buries the right article
(article-recall@200 ≈ 0.15) under provincial decisions (`QĐ-UBND`) and superseded
decrees.

## Architecture

| Stage | Module | What it does |
|---|---|---|
| A. Anchor | [`anchor_documents`](src/retrieval/doc_anchor.py) | RRF-fuse lexical+dense chunk hits into a **document** ranking, nudged by a soft authority prior (central/recent up, provincial/superseded down). |
| B. Harvest | [`harvest_articles`](src/retrieval/doc_anchor.py) | Within anchored docs, keep the best chunk per `Điều X` → article-candidate pool. |
| C. Rerank | `BAAI/bge-reranker-v2-m3` | Cross-encoder reranks **article passages** — lifts the right article above siblings. |
| D. Select | [`select_articles`](src/retrieval/article_select.py) | Authority suppression + F2-optimal K (recall floor 1, admit confident 2nd within a margin, cap `max_k`). |

Generation is intentionally skipped: the grader reads `relevant_articles`, so the
entire budget goes to retrieval + rerank. The `answer` field is emitted empty to
satisfy the submission schema.


## 1. Setup — clone repo, attach Dataset, install deps, configure

In [ ]:
# ===== Configuration (edit these) =====================================
GITHUB_REPO = "https://github.com/vkb0205/Road2AI_ApplePie.git"
REPO_BRANCH = "main"
REPO_DIR    = "/kaggle/working/Road2AI_ApplePie"

# Kaggle Dataset that holds the Stage-6 artifacts (same one the decomposition
# notebook uses). Auto-mounted under /kaggle/input/<dataset-name>/.
KAGGLE_DATASET = "vkb0205/stage6-data"          # <-- edit to YOUR dataset slug
DATA_DIR = f"/kaggle/input/{KAGGLE_DATASET.split('/')[-1]}"
DEV_DIR  = f"{REPO_DIR}/dev_set"

# --- Pipeline switches -------------------------------------------------
USE_DENSE  = True            # FAISS + BGE-m3 query encoder (GPU). REQUIRED for recall.
USE_RERANK = True            # cross-encoder BAAI/bge-reranker-v2-m3 (GPU). REQUIRED.
FTS_MODE   = "bm25_ranked"   # 'bm25_ranked' (stronger) or 'fts_fast'
GPU_ID     = 0

# --- Retrieval knobs (DocAnchorConfig) --------------------------------
TOP_BM25          = 150      # lexical candidates (wide intake for recall)
TOP_DENSE         = 150      # dense candidates
RRF_K             = 60       # RRF smoothing at the document-anchor stage
TOP_DOCS          = 10       # anchored documents kept
PER_DOC_ARTICLES  = 6        # max distinct articles harvested per doc
RERANK_POOL       = 40       # max article candidates sent to the reranker

# --- Selection knobs (SelectConfig) -----------------------------------
DROP_PROVINCIAL   = True     # hard-drop QĐ-UBND / QĐ-UB issuers
MAX_K             = 2        # most dev questions need 1, four need 2
MIN_K             = 1        # recall floor
REL_MARGIN        = 0.15     # admit 2nd article within 15% of the top score
ABS_MARGIN        = 0.12     # ...and within 0.12 absolute

# --- Output ------------------------------------------------------------
RESULTS_PATH        = "/kaggle/working/results.json"
POOL_PATH           = "/kaggle/working/pool.json"
SUBMISSION_ZIP_PATH = "/kaggle/working/submission.zip"
# ======================================================================

# --- 0. GPU runtime check (fail fast) ---------------------------------
import subprocess, sys, os
from pathlib import Path

if USE_DENSE or USE_RERANK:
    gpu_ok = subprocess.run("nvidia-smi", shell=True,
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
    if not gpu_ok:
        raise SystemExit("❌ No GPU detected. Set Accelerator → GPU, or disable USE_DENSE/USE_RERANK.")
    !nvidia-smi -L
    print("[gpu] GPU runtime confirmed.")

# --- 1a. Clone the retrieval source code ------------------------------
if not Path(REPO_DIR).exists():
    print(f"[clone] {GITHUB_REPO} -> {REPO_DIR}")
    !git clone --depth 1 -b {REPO_BRANCH} {GITHUB_REPO} {REPO_DIR}
else:
    print(f"[clone] {REPO_DIR} already present; pulling latest")
    !cd {REPO_DIR} && git pull --ff-only

SRC = Path(REPO_DIR) / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)   # so `import dev_set.eval` works
print("[src on path]", SRC)

# --- 1b. Confirm the Stage-6 Dataset is attached ----------------------
DATA = Path(DATA_DIR)
DEV  = Path(DEV_DIR)
print("[DATA_DIR exists]", DATA.exists(), DATA)
if DATA.exists():
    for p in sorted(DATA.iterdir()):
        print("   ", p.name)
print("[DEV_DIR  exists]", DEV.exists(), DEV)

# --- 1c. Install dependencies -----------------------------------------
!pip -q install pandas pyarrow networkx pyyaml python-dotenv psutil
if USE_DENSE or USE_RERANK:
    # torch is preinstalled on the Kaggle GPU image; FlagEmbedding + faiss may not be.
    try:
        import FlagEmbedding  # noqa: F401
    except Exception:
        !pip -q install FlagEmbedding
    try:
        import faiss  # noqa: F401
    except Exception:
        !pip -q install faiss-gpu-cu12 || pip -q install faiss-cpu
print("[deps] ready")


## 2. Build the recall-first pipeline

Wires the real Stage-6 bundle (FTS5 + FAISS + BGE-m3 + cross-encoder) into the
unit-tested orchestration in [`doc_anchor.py`](src/retrieval/doc_anchor.py) and the
selector in [`article_select.py`](src/retrieval/article_select.py).

In [ ]:
from retrieval.kaggle_pipeline import build_pipeline, run_dev_set, dump_json
from retrieval.doc_anchor import DocAnchorConfig
from retrieval.article_select import SelectConfig

anchor_cfg = DocAnchorConfig(
    top_bm25=TOP_BM25, top_dense=(TOP_DENSE if USE_DENSE else 0),
    rrf_k=RRF_K, top_docs=TOP_DOCS,
    per_doc_articles=PER_DOC_ARTICLES, rerank_pool=RERANK_POOL,
)
select_cfg = SelectConfig(
    drop_provincial=DROP_PROVINCIAL, max_k=MAX_K, min_k=MIN_K,
    rel_margin=REL_MARGIN, abs_margin=ABS_MARGIN,
)

pipe = build_pipeline(
    DATA_DIR,
    use_dense=USE_DENSE, use_rerank=USE_RERANK,
    fts_mode=FTS_MODE, gpu_id=GPU_ID,
    anchor_cfg=anchor_cfg, select_cfg=select_cfg,
)
print("[pipeline] built  | dense=%s rerank=%s" % (USE_DENSE, USE_RERANK))

# Smoke test on one question.
_demo = pipe.answer("Thủ tục đăng ký doanh nghiệp lần đầu bao gồm những bước nào?")
for a in _demo:
    print(f"   {a.dieu}  <- {a.law_id}  (score={a.final_score:.3f}, support={a.n_support})")


## 3. Run the dev set — submission + tunable candidate pool

`run_dev_set` returns the grader-format `submission` and a `pool` dump of the
pre-selection reranked candidates, so the K-selection policy can be retuned
offline **without** re-running the GPU legs.

In [ ]:
import time
_t0 = time.time()
submission, pool = run_dev_set(pipe, f"{DEV_DIR}/ground_truth.json")
print(f"[run] {len(submission)} questions in {time.time()-_t0:.1f}s")

dump_json(submission, RESULTS_PATH)
dump_json(pool, POOL_PATH)
print("[write]", RESULTS_PATH)
print("[write]", POOL_PATH)

# Free the GPU before scoring.
pipe.close()
try:
    import torch, gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass


## 4. Score F2 + per-question diagnostics + offline policy tuning

In [ ]:
# Grader-faithful F2 (reads relevant_articles, number-only normalisation).
!cd {REPO_DIR} && python dev_set/f2_diagnostics.py \
    --ground-truth dev_set/ground_truth.json \
    --predictions {RESULTS_PATH}

print("\n=== Grid-search the selection policy on the dumped pool (no GPU) ===")
# Recall ceiling tells you which lever to pull: low ceiling -> widen retrieval
# (TOP_DENSE / PER_DOC_ARTICLES); high ceiling but low F2 -> tune SelectConfig.
!cd {REPO_DIR} && python dev_set/tune_selector.py \
    --pool {POOL_PATH} \
    --ground-truth dev_set/ground_truth.json \
    --grid --top 8


## 5. Validate + package `submission.zip`

Validates `results.json` against the grader contract (record schema + id
coverage) and packages it with arcname `results.json`.

In [ ]:
import json as _json, zipfile

def validate_submission(path, expected_records):
    path = Path(path)
    assert path.exists(), f"Missing results file: {path}"
    data = _json.loads(path.read_text(encoding="utf-8"))
    assert isinstance(data, list), "results.json must be a JSON list"
    expected_ids = {int(r["id"]) for r in expected_records}
    got_ids = {int(r["id"]) for r in data}
    assert expected_ids == got_ids, {
        "missing": sorted(expected_ids - got_ids)[:20],
        "extra": sorted(got_ids - expected_ids)[:20]}
    required = {"id", "question", "answer", "relevant_docs", "relevant_articles"}
    bad = [r.get("id") for r in data if set(r.keys()) != required]
    assert not bad, f"Records with wrong schema: {bad[:10]}"
    for r in data[:5]:
        assert isinstance(r["relevant_docs"], list)
        assert isinstance(r["relevant_articles"], list)
    print({"records": len(data), "schema_ok": True, "ids_ok": True})

validate_submission(RESULTS_PATH, submission)

with zipfile.ZipFile(SUBMISSION_ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(RESULTS_PATH, arcname="results.json")
print(f"results written   : {RESULTS_PATH}")
print(f"submission written: {SUBMISSION_ZIP_PATH}")
